# ESS Battery Cycle Life Prediction

이 노트북은 **초기 사이클 기반 파생 피처**를 활용해 배터리의 **총 수명(`cycle_life`)**을 예측하는 회귀 실험입니다.

## 사용 모델
- **LightGBM Regressor (`LGBMRegressor`)**

## 적용 기법
- **EDA 기반 Feature Engineering**
- **Batch 1 학습 / Batch 2 테스트 분리**
- **Batch 1 내부 Hold-out Validation**
- **`log1p` 타겟 변환**
- **결측치 대체 (`SimpleImputer`)**
- **Optuna 하이퍼파라미터 튜닝**
- **K-Fold Cross Validation 재평가**
- **Feature Importance 분석**
- **Feature Set별 성능 비교**

## 실험 흐름
1. 실험 설정 및 feature set 선택  
2. 평가 지표(MAPE) 정의  
3. 데이터 로드 및 feature 유효성 확인  
4. Batch 간 분포 비교  
5. 전처리 및 학습/검증 데이터 분리  
6. Optuna 기반 튜닝  
7. CV 재평가  
8. 최종 모델 학습 및 Valid/Test 평가  
9. Feature importance 확인  
10. 최종 성능 리포트 정리  
11. 오차 큰 샘플 점검  
12. 여러 feature set 비교


## 0. 라이브러리 로드

모델 학습, 하이퍼파라미터 탐색, 데이터 전처리와 평가에 필요한 라이브러리를 불러옵니다.
특히 이 실험의 핵심 모델은 `LightGBMRegressor`이며, 튜닝에는 `Optuna`를 사용합니다.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.impute import SimpleImputer
from lightgbm import LGBMRegressor
import optuna

## 1. 실험 설정(Config)

이 셀에서는 다음을 설정합니다.

- 타겟 변수: `cycle_life`
- 비교 기준 성능: `TARGET_SCORE = 9.1`
- 랜덤 시드, Optuna trial 수, KFold 분할 수
- 사용할 feature set (`SELECTED_SET`)
- Batch 1 / Batch 2 데이터 경로 자동 탐색

### 선택된 feature set
현재 기본값은 **`trimmed_recovered_set`** 입니다.  
즉, 논문 feature 5개만 쓰는 것이 아니라, EDA와 복원/정리 과정을 거친 확장 feature 집합을 사용합니다.


In [2]:
# =========================
# 1. Config
# =========================
TARGET       = "cycle_life"
TARGET_SCORE = 9.1
RANDOM_STATE = 42
N_TRIALS     = 100
N_SPLITS     = 5

SELECTED_SET = "trimmed_recovered_set"
# 후보:
# "paper_5"
# "trimmed_recovered_set"
# "recovered_set"
# "all_numeric"

FEATURE_SETS = {
    "paper_5": [
        "delta_log_var",
        "delta_max",
        "QD_cv",
        "effective_C",
        "temp_ir_interaction",
    ],

    "trimmed_recovered_set": [
        "C_rate_1",
        "ratio",
        "C_rate_2",
        "effective_C",
        "QD_mean",
        "QD_cv",
        "IR_mean",
        "Tavg_mean",
        "temp_ir_interaction",
        "delta_log_var",
        "delta_range",
        "delta_area",
        "delta_v_2_8",
        "delta_v_3_0",
        "delta_28_30_mean",
    ],

    "recovered_set": [
        "C_rate_1",
        "ratio",
        "C_rate_2",
        "effective_C",
        "QD_mean",
        "QD_cv",
        "IR_mean",
        "IR_cv",
        "Tavg_mean",
        "temp_ir_interaction",
        "delta_log_var",
        "delta_max",
        "delta_range",
        "delta_area",
        "delta_v_2_8",
        "delta_v_3_0",
        "delta_28_30_mean",
    ],

    "all_numeric": None,
}

BATCH1_CANDIDATES = [
    Path("batch1_model_df.csv"),
    Path("data/processed/batch1_model_df.csv"),
    Path("/Users/skax/skala/ess-battery-project/data/processed/batch1_model_df.csv"),
]
BATCH2_CANDIDATES = [
    Path("batch2_model_df.csv"),
    Path("data/processed/batch2_model_df.csv"),
    Path("/Users/skax/skala/ess-battery-project/data/processed/batch2_model_df.csv"),
]

def resolve_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return str(p)
    raise FileNotFoundError(
        "CSV 파일을 찾지 못했습니다.\n"
        f"batch1 후보: {[str(p) for p in BATCH1_CANDIDATES]}\n"
        f"batch2 후보: {[str(p) for p in BATCH2_CANDIDATES]}"
    )

BATCH1_PATH = resolve_existing_path(BATCH1_CANDIDATES)
BATCH2_PATH = resolve_existing_path(BATCH2_CANDIDATES)

print("BATCH1_PATH =", BATCH1_PATH)
print("BATCH2_PATH =", BATCH2_PATH)
print("SELECTED_SET =", SELECTED_SET)

BATCH1_PATH = /Users/skax/skala/ess-battery-project/data/processed/batch1_model_df.csv
BATCH2_PATH = /Users/skax/skala/ess-battery-project/data/processed/batch2_model_df.csv
SELECTED_SET = trimmed_recovered_set


## 2. 평가 지표 정의

회귀 성능 평가는 **MAPE(Mean Absolute Percentage Error)** 로 수행합니다.

- 값이 낮을수록 예측 오차가 작음
- 결과 해석이 직관적이라 배터리 수명 예측 성능 비교에 적합


In [3]:
# =========================
# 2. Metric
# =========================
def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

## 3. 데이터 로드

Batch 1, Batch 2 데이터를 불러오고 다음을 확인합니다.

- 데이터 shape
- 선택한 feature set이 실제 데이터에 존재하는지
- `all_numeric` 후보 생성

여기서 실험에 사용할 최종 feature 목록이 결정됩니다.


In [5]:
# =========================
# 3. 데이터 로드
# =========================
batch1 = pd.read_csv(BATCH1_PATH)
batch2 = pd.read_csv(BATCH2_PATH)

print(f"batch1 shape: {batch1.shape}")
print(f"batch2 shape: {batch2.shape}")

exclude_cols = {"cell_id", TARGET}
FEATURE_SETS["all_numeric"] = [
    c for c in batch1.select_dtypes(include=[np.number]).columns
    if c not in exclude_cols
]

FEATURES = FEATURE_SETS[SELECTED_SET]

missing_b1 = [c for c in FEATURES if c not in batch1.columns]
missing_b2 = [c for c in FEATURES if c not in batch2.columns]

if missing_b1:
    raise ValueError(f"Batch1에 없는 feature: {missing_b1}")
if missing_b2:
    raise ValueError(f"Batch2에 없는 feature: {missing_b2}")

print("\n사용 가능한 전체 피처 수:", len(FEATURE_SETS["all_numeric"]))
print("SELECTED_SET:", SELECTED_SET)
print("사용 feature 수:", len(FEATURES))
print("사용 feature 목록:", FEATURES)

batch1 shape: (46, 33)
batch2 shape: (39, 33)

사용 가능한 전체 피처 수: 31
SELECTED_SET: trimmed_recovered_set
사용 feature 수: 15
사용 feature 목록: ['C_rate_1', 'ratio', 'C_rate_2', 'effective_C', 'QD_mean', 'QD_cv', 'IR_mean', 'Tavg_mean', 'temp_ir_interaction', 'delta_log_var', 'delta_range', 'delta_area', 'delta_v_2_8', 'delta_v_3_0', 'delta_28_30_mean']


## 4. EDA: Batch 간 분포 비교

학습 데이터(Batch 1)와 테스트 데이터(Batch 2) 사이에서
선택된 feature들의 평균/표준편차 차이를 확인합니다.

### 목적
- 배치 간 분포 차이(domain shift) 확인
- 일반화 성능 저하 가능성 사전 점검
- 어떤 feature가 테스트 성능을 흔들 수 있는지 해석 근거 확보


In [6]:
# =========================
# 4. EDA — Batch 간 분포 비교
# =========================
print("=" * 55)
print(f"{'Feature':<22} {'B1 mean':>9} {'B1 std':>8} {'B2 mean':>9} {'B2 std':>8} {'Δmean%':>9}")
print("=" * 55)

dist_rows = []

for feat in FEATURES:
    b1_mean = batch1[feat].mean()
    b1_std  = batch1[feat].std()
    b2_mean = batch2[feat].mean()
    b2_std  = batch2[feat].std()
    delta   = abs(b2_mean - b1_mean) / (abs(b1_mean) + 1e-9) * 100
    flag    = " ←" if delta > 30 else ""

    print(f"{feat:<22} {b1_mean:>9.3f} {b1_std:>8.3f} {b2_mean:>9.3f} {b2_std:>8.3f} {delta:>8.1f}%{flag}")

    dist_rows.append({
        "feature": feat,
        "b1_mean": b1_mean,
        "b2_mean": b2_mean,
        "mean_shift_pct": delta,
        "b1_std": b1_std,
        "b2_std": b2_std,
    })

dist_df = pd.DataFrame(dist_rows).sort_values("mean_shift_pct", ascending=False)

print("\n[mean shift 상위 5개]")
print(dist_df.head(5).to_string(index=False))

print("\n[cycle_life 분포]")
print(f"  Batch1: mean={batch1[TARGET].mean():.0f}, std={batch1[TARGET].std():.0f}, "
      f"min={batch1[TARGET].min():.0f}, max={batch1[TARGET].max():.0f}")
print(f"  Batch2: mean={batch2[TARGET].mean():.0f}, std={batch2[TARGET].std():.0f}, "
      f"min={batch2[TARGET].min():.0f}, max={batch2[TARGET].max():.0f}")
print(f"  skewness — Batch1: {batch1[TARGET].skew():.2f}, Batch2: {batch2[TARGET].skew():.2f}")

Feature                  B1 mean   B1 std   B2 mean   B2 std    Δmean%
C_rate_1                   5.878    1.203     5.031    0.614     14.4%
ratio                     51.522   19.461    50.103   21.757      2.8%
C_rate_2                   3.583    0.584     4.568    0.671     27.5%
effective_C                2.833    0.709     2.524    1.069     10.9%
QD_mean                    1.082    0.007     1.095    0.011      1.2%
QD_cv                      0.001    0.001     0.004    0.004    337.4% ←
IR_mean                    0.017    0.000     0.014    0.006     15.8%
Tavg_mean                 32.617    0.917    32.898    1.145      0.9%
temp_ir_interaction        0.539    0.020     0.458    0.200     14.9%
delta_log_var             -3.923    0.378    -3.596    0.394      8.3%
delta_range                0.036    0.013     0.053    0.018     46.1% ←
delta_area                 0.024    0.010     0.038    0.016     55.9% ←
delta_v_2_8               -0.030    0.012    -0.044    0.017     47.6% 

## 5. 전처리 및 데이터 분리

이 단계에서 다음 작업을 수행합니다.

- `X`, `y` 분리
- 타겟 `cycle_life`에 대해 **`log1p` 변환**
- Batch 1 내부를 Train / Valid로 분리
- 결측치 확인 및 **`SimpleImputer`** 로 대체

### 왜 `log1p`를 쓰는가?
배터리 수명은 분포가 치우칠 수 있으므로,
로그 변환을 통해 극단값 영향을 줄이고 학습 안정성을 높이기 위함입니다.


In [7]:
# =========================
# 5. X, y 분리 + NaN 처리 + 타겟 log1p 변환
# =========================
X_raw = batch1[FEATURES].copy()
y_raw = batch1[TARGET].copy()
y_log = np.log1p(y_raw)

X_test_raw = batch2[FEATURES].copy()
y_test_raw = batch2[TARGET].copy()

nan_b1 = X_raw.isna().sum()
nan_b2 = X_test_raw.isna().sum()

print("Batch1 NaN:", nan_b1[nan_b1 > 0].to_dict())
print("Batch2 NaN:", nan_b2[nan_b2 > 0].to_dict())

X_tr_raw, X_val_raw, y_train_log, y_valid_log, y_train_raw, y_valid_raw = train_test_split(
    X_raw,
    y_log,
    y_raw,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_tr_raw),
    columns=FEATURES,
    index=X_tr_raw.index
)

X_valid = pd.DataFrame(
    imputer.transform(X_val_raw),
    columns=FEATURES,
    index=X_val_raw.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=FEATURES,
    index=X_test_raw.index
)

print(f"\nX_train: {X_train.shape}, NaN: {X_train.isna().sum().sum()}")
print(f"X_valid: {X_valid.shape}, NaN: {X_valid.isna().sum().sum()}")
print(f"X_test : {X_test.shape}, NaN: {X_test.isna().sum().sum()}")

Batch1 NaN: {}
Batch2 NaN: {}

X_train: (36, 15), NaN: 0
X_valid: (10, 15), NaN: 0
X_test : (39, 15), NaN: 0


## 6. Optuna 하이퍼파라미터 튜닝

`LightGBMRegressor`의 주요 하이퍼파라미터를
**보수적인 탐색 범위**에서 Optuna로 최적화합니다.

### 핵심 포인트
- 목적 함수: Validation MAPE 최소화
- 탐색 대상: `learning_rate`, `n_estimators`, `max_depth`, `num_leaves` 등
- 과적합을 피하기 위해 너무 공격적인 탐색 범위는 제한


In [8]:
# =========================
# 6. Optuna 튜닝 (보수적 범위)
# =========================
def objective(trial):
    params = {
        "objective": "regression",
        "metric": "None",
        "random_state": RANDOM_STATE,
        "verbosity": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 700),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "num_leaves": trial.suggest_int("num_leaves", 7, 20),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 12),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
    }

    model = LGBMRegressor(**params)

    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    fold_mapes = []

    for tr_idx, va_idx in cv.split(X_train):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train_log.iloc[tr_idx]
        y_va = y_train_log.iloc[va_idx]

        model.fit(X_tr, y_tr)

        pred_va_log = model.predict(X_va)
        pred_va = np.expm1(pred_va_log)
        true_va = np.expm1(y_va)

        fold_mapes.append(mape(true_va, pred_va))

    return np.mean(fold_mapes)

print(f"Optuna 탐색 시작 ({N_TRIALS} trials) ...")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
best_value = study.best_value

print("\nBest CV MAPE (원래 스케일, %):", round(best_value, 4))
print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-31 16:50:26,671] A new study created in memory with name: no-name-dc60cad0-6e53-4629-aae0-10c3ec3ecfbf


Optuna 탐색 시작 (100 trials) ...


Best trial: 0. Best value: 12.3762:   1%|          | 1/100 [00:00<00:42,  2.31it/s]

[I 2026-03-31 16:50:27,114] Trial 0 finished with value: 12.376161176238694 and parameters: {'learning_rate': 0.05441343973730855, 'n_estimators': 348, 'max_depth': 3, 'num_leaves': 17, 'min_child_samples': 5, 'reg_alpha': 0.5740188789859261, 'reg_lambda': 2.5823546202218117, 'subsample': 0.7681982894669627, 'colsample_bytree': 0.8560517064853448}. Best is trial 0 with value: 12.376161176238694.


Best trial: 1. Best value: 10.7853:   2%|▏         | 2/100 [00:00<00:43,  2.26it/s]

...

[I 2026-03-31 16:51:55,435] Trial 97 finished with value: 8.422446597248815 and parameters: {'learning_rate': 0.07977986492082642, 'n_estimators': 687, 'max_depth': 3, 'num_leaves': 13, 'min_child_samples': 6, 'reg_alpha': 0.010553250428544445, 'reg_lambda': 0.005262634748916518, 'subsample': 0.9297029190070498, 'colsample_bytree': 0.9289686071032983}. Best is trial 73 with value: 8.015929690114143.


Best trial: 73. Best value: 8.01593:  99%|█████████▉| 99/100 [01:29<00:01,  1.04s/it]

[I 2026-03-31 16:51:56,459] Trial 98 finished with value: 8.17055611997406 and parameters: {'learning_rate': 0.0742562052967332, 'n_estimators': 670, 'max_depth': 3, 'num_leaves': 11, 'min_child_samples': 5, 'reg_alpha': 0.0032337640080926148, 'reg_lambda': 0.0012468460153730958, 'subsample': 0.9715632867518721, 'colsample_bytree': 0.9443955349102036}. Best is trial 73 with value: 8.015929690114143.


Best trial: 73. Best value: 8.01593: 100%|██████████| 100/100 [01:30<00:00,  1.10it/s]

[I 2026-03-31 16:51:57,480] Trial 99 finished with value: 8.334842768953354 and parameters: {'learning_rate': 0.06416681725266017, 'n_estimators': 638, 'max_depth': 3, 'num_leaves': 12, 'min_child_samples': 5, 'reg_alpha': 0.006561208876586131, 'reg_lambda': 0.00181046663308223, 'subsample': 0.9536485557812727, 'colsample_bytree': 0.9518188126272394}. Best is trial 73 with value: 8.015929690114143.

Best CV MAPE (원래 스케일, %): 8.0159
Best params:
  learning_rate: 0.024528738247916453
  n_estimators: 700
  max_depth: 3
  num_leaves: 10
  min_child_samples: 5
  reg_alpha: 0.007106699768547838
  reg_lambda: 0.0013942988210320546
  subsample: 0.9396650910931561
  colsample_bytree: 0.9420251797260819


## 7. Batch 1 CV 재평가

Optuna로 찾은 best parameter를 사용해
Batch 1 학습 데이터 내부에서 **K-Fold Cross Validation**을 다시 수행합니다.

### 목적
- Hold-out 한 번의 결과에만 의존하지 않기
- 학습 데이터 내 평균적인 일반화 성능 확인
- 최종 리포트용 `Train (Batch 1 CV)` 성능 산출


In [9]:
# =========================
# 7. Train (Batch 1 CV) 재평가
# =========================
best_model_params = {
    "objective": "regression",
    "metric": "None",
    "random_state": RANDOM_STATE,
    "verbosity": -1,
    **best_params
}

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_fold_scores = []

for tr_idx, va_idx in cv.split(X_train):
    X_tr = X_train.iloc[tr_idx]
    X_va = X_train.iloc[va_idx]
    y_tr = y_train_log.iloc[tr_idx]
    y_va = y_train_log.iloc[va_idx]

    model = LGBMRegressor(**best_model_params)
    model.fit(X_tr, y_tr)

    pred_va_log = model.predict(X_va)
    pred_va = np.expm1(pred_va_log)
    true_va = np.expm1(y_va)

    cv_fold_scores.append(mape(true_va, pred_va))

train_cv_mape = np.mean(cv_fold_scores)

print("CV fold MAPE (원래 스케일, %):", np.round(cv_fold_scores, 4))
print(f"Train (Batch 1 CV) MAPE: {train_cv_mape:.4f}")

CV fold MAPE (원래 스케일, %): [ 9.33   10.2406  7.1229  5.8589  7.5273]
Train (Batch 1 CV) MAPE: 8.0159


## 8. 최종 모델 학습 및 Valid/Test 평가

최적 하이퍼파라미터로 최종 LightGBM 모델을 학습하고,
다음 성능을 계산합니다.

- **Valid MAPE**: Batch 1 hold-out
- **Test MAPE**: Batch 2

예측은 로그 스케일에서 수행한 뒤 `expm1`으로 원래 스케일로 복원합니다.


In [10]:
# =========================
# 8. Final model 학습 + Valid/Test 평가
# =========================
lgbm_final = LGBMRegressor(**best_model_params)
lgbm_final.fit(X_train, y_train_log)

valid_pred_log = lgbm_final.predict(X_valid)
test_pred_log  = lgbm_final.predict(X_test)

valid_pred = np.expm1(valid_pred_log)
test_pred  = np.expm1(test_pred_log)

valid_mape = mape(y_valid_raw, valid_pred)
test_mape  = mape(y_test_raw, test_pred)

print(f"Valid MAPE: {valid_mape:.4f} %")
print(f"Test  MAPE: {test_mape:.4f} %")

Valid MAPE: 7.1767 %
Test  MAPE: 29.6541 %


## 9. Feature Importance 분석

최종 LightGBM 모델의 feature importance를 확인합니다.

### 목적
- 실제 예측에 크게 기여한 변수 파악
- 논문 기반 핵심 feature가 상위권에 들어오는지 확인
- EDA에서 설계한 feature가 유효했는지 해석 근거 확보


In [11]:
# =========================
# 9. Feature Importance
# =========================
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": lgbm_final.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 10 Feature Importance (LightGBM):")
print(importance.head(10).to_string(index=False))

paper_feats = FEATURE_SETS["paper_5"]
top10_feats = set(importance.head(10)["feature"].tolist())
paper_in_top10 = top10_feats & set(paper_feats)

print(f"\n논문 피처 중 Top10 포함: {sorted(paper_in_top10)}")

Top 10 Feature Importance (LightGBM):
            feature  importance
          Tavg_mean         365
            QD_mean         340
              QD_cv         283
        delta_v_3_0         276
        delta_v_2_8         212
        effective_C         164
           C_rate_2         153
        delta_range         136
temp_ir_interaction         130
              ratio         111

논문 피처 중 Top10 포함: ['QD_cv', 'effective_C', 'temp_ir_interaction']


## 10. 최종 성능 리포트

최종적으로 아래 항목들을 표 형태로 정리합니다.

- Train (Batch 1 CV)
- Valid (Batch 1 Hold-out)
- Test (Batch 2)
- Gap (Train-Valid)
- Gap (Valid-Test)
- Gap (Target-Test)

### 해석 포인트
Gap이 크면 과적합, 분포 차이, feature 한계 등을 의심할 수 있습니다.


In [12]:
# =========================
# 10. 최종 성능 리포트
# =========================
gap_train_valid = valid_mape - train_cv_mape
gap_valid_test  = test_mape - valid_mape
gap_target_test = test_mape - TARGET_SCORE

result_df = pd.DataFrame({
    "Index": [
        "Train (Batch 1 CV)",
        "Valid (Batch 1 Hold-out)",
        "Test (Batch 2)",
        "Gap (Train-Valid)",
        "Gap (Valid-Test)",
        "Gap (Target-Test)"
    ],
    "MAPE (%)": [
        train_cv_mape,
        valid_mape,
        test_mape,
        gap_train_valid,
        gap_valid_test,
        gap_target_test
    ]
})

print(result_df.round(4).to_string(index=False))

                   Index  MAPE (%)
      Train (Batch 1 CV)    8.0159
Valid (Batch 1 Hold-out)    7.1767
          Test (Batch 2)   29.6541
       Gap (Train-Valid)   -0.8392
        Gap (Valid-Test)   22.4774
       Gap (Target-Test)   20.5541


## 11. Test 오차 큰 샘플 확인

테스트셋에서 APE가 큰 샘플을 확인합니다.

### 목적
- 어떤 셀에서 예측이 크게 빗나갔는지 파악
- 이상치, 배치 차이, 특정 charging policy 영향 등을 추정
- 성능 한계 해석 근거 확보


In [13]:
# =========================
# 11. Test 오차 큰 샘플 확인
# =========================
test_result = pd.DataFrame({
    "y_true": y_test_raw.values,
    "y_pred": test_pred
})

test_result["ape"] = np.abs((test_result["y_true"] - test_result["y_pred"]) / test_result["y_true"]) * 100

print(test_result.sort_values("ape", ascending=False).head(10).to_string(index=False))

 y_true     y_pred       ape
  392.0 648.826154 65.516876
  408.0 669.188159 64.016706
  393.0 615.063242 56.504642
  412.0 605.839649 47.048458
  449.0 658.642600 46.691002
  396.0 570.842528 44.152154
  425.0 602.621386 41.793267
  452.0 624.561083 38.177231
  416.0 572.231029 37.555536
  424.0 575.948044 35.836803


## 12. Feature Set 빠른 비교

여러 feature set을 동일한 흐름으로 빠르게 비교합니다.

비교 대상:
- `paper_5`
- `trimmed_recovered_set`
- `recovered_set`
- `all_numeric`

### 목적
- 논문 feature 5개만 쓸 때와 확장 feature set의 차이 확인
- 과도한 변수 추가가 성능을 악화시키는지 점검
- 현재 실습에 가장 적합한 feature set 선택


In [14]:
# =========================
# 12. Feature set 빠른 비교
# =========================
compare_sets = ["paper_5", "trimmed_recovered_set", "recovered_set", "all_numeric"]

compare_rows = []

for set_name in compare_sets:
    feats = FEATURE_SETS[set_name]

    X_raw_cmp = batch1[feats].copy()
    y_raw_cmp = batch1[TARGET].copy()
    y_log_cmp = np.log1p(y_raw_cmp)

    X_test_raw_cmp = batch2[feats].copy()
    y_test_cmp = batch2[TARGET].copy()

    X_tr_raw_cmp, X_val_raw_cmp, y_train_log_cmp, y_valid_log_cmp, y_train_raw_cmp, y_valid_raw_cmp = train_test_split(
        X_raw_cmp,
        y_log_cmp,
        y_raw_cmp,
        test_size=0.2,
        random_state=RANDOM_STATE,
    )

    imp = SimpleImputer(strategy="median")

    X_train_cmp = pd.DataFrame(imp.fit_transform(X_tr_raw_cmp), columns=feats)
    X_valid_cmp = pd.DataFrame(imp.transform(X_val_raw_cmp), columns=feats)
    X_test_cmp  = pd.DataFrame(imp.transform(X_test_raw_cmp), columns=feats)

    model_cmp = LGBMRegressor(**best_model_params)
    model_cmp.fit(X_train_cmp, y_train_log_cmp)

    valid_pred_cmp = np.expm1(model_cmp.predict(X_valid_cmp))
    test_pred_cmp  = np.expm1(model_cmp.predict(X_test_cmp))

    valid_mape_cmp = mape(y_valid_raw_cmp, valid_pred_cmp)
    test_mape_cmp  = mape(y_test_cmp, test_pred_cmp)

    compare_rows.append({
        "feature_set": set_name,
        "n_features": len(feats),
        "valid_mape": valid_mape_cmp,
        "test_mape": test_mape_cmp,
        "gap_valid_test": test_mape_cmp - valid_mape_cmp
    })

compare_df = pd.DataFrame(compare_rows).sort_values("test_mape")
print(compare_df.round(4).to_string(index=False))

          feature_set  n_features  valid_mape  test_mape  gap_valid_test
          all_numeric          31      2.3846    24.3828         21.9982
              paper_5           5      9.4166    28.4202         19.0036
trimmed_recovered_set          15      7.1767    29.6541         22.4774
        recovered_set          17      7.3898    31.7842         24.3944
